In [ ]:
import json
import os
import math
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display
from rdkit import RDLogger
# 屏蔽 RDKit 的警告日志
RDLogger.DisableLog('rdApp.*')

# 定义文件路径 (根据您的目录结构)
base_dir = "processed_data"
# 您可以在这里切换想看的文件： fragment_vocabulary_filter.json / fragment_vocabulary_puppy.json / fragment_vocabulary_frame.json / fragment_vocabulary_linker.json
target_filename = "fragment_vocabulary_valid.json" 

file_path = os.path.join(base_dir, target_filename)

print(f"Target File: {os.path.abspath(file_path)}")

In [ ]:
def visualize_fragments_in_batches(json_path, batch_size=20):
    # 1. 读取 JSON
    if not os.path.exists(json_path):
        print(f"Error: File not found at {json_path}")
        return

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    valid_mols = []
    legends = []
    failed_smiles = []
    special_tokens = [] # 像 <UNK> 这种
    
    print(f"Total entries in file: {len(data)}")
    print("-" * 30)
    
    # 2. 处理 SMILES
    for raw_smile in data.keys():
        # 跳过特殊标记 (如 <UNK>)
        if raw_smile.startswith("<") and raw_smile.endswith(">"):
            special_tokens.append(raw_smile)
            continue
            
        # --- 关键步骤：移除 '*' 标记 ---
        # 很多片段库会用 * 表示连接点，RDKit有时无法直接解析带 * 的孤立片段
        clean_smile = raw_smile.replace("*", "")
        
        # 如果移除 * 后为空字符串 (例如原片段就是 "*")
        if not clean_smile:
            failed_smiles.append(f"{raw_smile} (Empty after cleaning)")
            continue

        # --- 尝试转换为 RDKit 分子对象 ---
        # sanitize=False 可以尝试读取一些稍微有点问题的分子，
        # 但为了绘图美观，通常还是默认 sanitize=True。
        # 如果这里报错，说明分子结构确实无法被 RDKit 理解。
        mol = Chem.MolFromSmiles(clean_smile)
        
        if mol:
            valid_mols.append(mol)
            # 在图下面显示原始的 smile (带 * 的)，方便您对照
            legends.append(raw_smile) 
        else:
            failed_smiles.append(raw_smile)

    # 3. 生成网格图 (分页)
    total_valid = len(valid_mols)
    num_batches = math.ceil(total_valid / batch_size)
    
    print(f"Successfully parsed: {total_valid} molecules")
    print(f"Generating {num_batches} grid images (20 mols per image)...")
    print("=" * 30)

    for i in range(0, total_valid, batch_size):
        batch_mols = valid_mols[i : i + batch_size]
        batch_legends = legends[i : i + batch_size]
        
        # 计算当前是第几组
        batch_num = (i // batch_size) + 1
        print(f"Batch {batch_num} / {num_batches} (Index {i} - {min(i+batch_size, total_valid)})")
        
        # 生成图片
        # molsPerRow=5: 一行5个，一共4行 (5*4=20)
        img = Draw.MolsToGridImage(
            batch_mols, 
            molsPerRow=5, 
            subImgSize=(200, 200), 
            legends=batch_legends,
            returnPNG=False 
        )
        display(img)
        print("\n") # 空行隔开

    # 4. 日志输出：无法显示的分子
    print("=" * 30)
    print("LOG: Skipped / Failed Molecules")
    print("-" * 30)
    
    if special_tokens:
        print(f"Special Tokens (Skipped, {len(special_tokens)}):")
        print(special_tokens)
        print("-" * 10)
        
    if failed_smiles:
        print(f"Parse Errors (Failed to visualize, {len(failed_smiles)}):")
        for s in failed_smiles:
            print(f"  - {s}")
    else:
        print("No parse errors found.")

In [ ]:
# 运行函数
visualize_fragments_in_batches(file_path, batch_size=20)